In [18]:
# !pip install "protobuf==3.20.3" -q

ERROR: Could not find a version that satisfies the requirement protobuf==3.20.3 (from versions: none)
ERROR: No matching distribution found for protobuf==3.20.3


In [19]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import polars as pl
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression 
import xgboost as xgb
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
import pandas.api.types
from itertools import product
import warnings
import logging
import lightgbm as lgb
import platform
import sys
from contextlib import contextmanager 

# --- Setup and Logging ---
warnings.filterwarnings('ignore')
warnings.filterwarnings('ignore', module='lightgbm')

os.environ['CATBOOST_DRIVER_COMPATIBLE'] = '1'
os.environ['CATBOOST_QUIET_MODE'] = '1'

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')
logger = logging.getLogger(__name__)

DATA_PATH = Path('/kaggle/input/hull-tactical-market-prediction/')


In [20]:
# ===========================================================================
# WARNING SUPPRESSION CONTEXT MANAGER
# ===========================================================================
@contextmanager
def suppress_stderr():
    """Temporarily redirect stderr to devnull to suppress native C warnings."""
    original_stderr = sys.stderr
    try:
        # Redirect stderr to /dev/null
        with open(os.devnull, 'w') as f:
            sys.stderr = f
            yield
    finally:
        # Restore original stderr
        sys.stderr = original_stderr

In [21]:
# ===========================================================================
# OFFICIAL KAGGLE METRIC (EXACT COPY)
# ===========================================================================
MIN_INVESTMENT = 0
MAX_INVESTMENT = 2

def score(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str = 'row_id') -> float:
    if not pd.api.types.is_numeric_dtype(submission['prediction']):
        raise ValueError('Predictions must be numeric')

    sol = solution.copy()
    sol['position'] = submission['prediction'].values

    if sol['position'].max() > MAX_INVESTMENT:
        raise ValueError(f'Position exceeds {MAX_INVESTMENT}')
    if sol['position'].min() < MIN_INVESTMENT:
        raise ValueError(f'Position below {MIN_INVESTMENT}')

    sol['strategy_returns'] = sol['risk_free_rate'] * (1 - sol['position']) + sol['position'] * sol['forward_returns']
    strategy_excess = sol['strategy_returns'] - sol['risk_free_rate']
    strategy_cum = (1 + strategy_excess).prod()
    strategy_mean = strategy_cum ** (1 / len(sol)) - 1
    strategy_std = sol['strategy_returns'].std()
    trading_days = 252

    if strategy_std == 0:
        return 0.0
    sharpe = strategy_mean / strategy_std * np.sqrt(trading_days)

    strategy_vol = float(strategy_std * np.sqrt(trading_days) * 100)

    market_excess = sol['forward_returns'] - sol['risk_free_rate']
    market_cum = (1 + market_excess).prod()
    market_mean = market_cum ** (1 / len(sol)) - 1
    market_std = sol['forward_returns'].std()
    market_vol = float(market_std * np.sqrt(trading_days) * 100)

    if market_vol == 0:
        return 0.0

    excess_vol = max(0, strategy_vol / market_vol - 1.2)
    vol_penalty = 1 + excess_vol

    return_gap = max(0, (market_mean - strategy_mean) * 100 * trading_days)
    return_penalty = 1 + (return_gap ** 2) / 100

    adjusted_sharpe = sharpe / (vol_penalty * return_penalty)
    return min(float(adjusted_sharpe), 1_000_000)


In [22]:
# ===========================================================================
# FEATURE ENGINEERING FUNCTION (POLARS - RESTORED ORIGINAL SET)
# ===========================================================================
def create_features(df: pl.DataFrame) -> pl.DataFrame:
    df_copy = df.clone()
    potential_base_feature_prefixes = ('M','E','I','P','V','S')
    all_potential_features = [
        c for c in df_copy.columns
        if c.startswith(potential_base_feature_prefixes) and c != 'market_forward_excess_returns'
    ]
    casting_expressions = []
    for c in all_potential_features:
        casting_expressions.append(
            pl.col(c).cast(pl.Float64, strict=False).alias(c)
        )
    if casting_expressions:
        df_copy = df_copy.with_columns(casting_expressions)
    base_features = [
        c for c in all_potential_features
        if df_copy.schema.get(c) in pl.NUMERIC_DTYPES
    ]
    expressions = []

    # --- 1. Lags (Original: 4 Lags) ---
    for c in base_features:
        for lag in [1, 2, 5, 10]:
            expressions.append(
                pl.col(c).shift(lag).over('date_id').fill_null(0).alias(f'{c}_L{lag}')
            )
    
    # --- 2. Rolling Window Features (Original: 2 Windows, 3 Stats) ---
    for c in base_features:
        for w in [5, 10]:
            expressions.append(
                pl.col(c).rolling_mean(window_size=w, min_periods=1).over('date_id').fill_null(0).alias(f'{c}_RMean{w}')
            )
            expressions.append(
                pl.col(c).rolling_std(window_size=w, min_periods=1).over('date_id').fill_nan(0).fill_null(0).alias(f'{c}_RStd{w}')
            )
            expressions.append(
                pl.col(c).rolling_max(window_size=w, min_periods=1).over('date_id').fill_null(0).alias(f'{c}_RMax{w}')
            )
            
    df_copy = df_copy.with_columns(expressions)
    expressions = []

    # --- 3. Rank and Z-Score ---
    for c in base_features:
        expressions.append(
            pl.col(c).rank(method='min').over('date_id').fill_null(0).alias(f'{c}_RANK')
        )
        mean_c = pl.col(c).mean().over('date_id')
        std_c  = pl.col(c).std().over('date_id')
        std_c_safe_expr = pl.when(pl.col(c).is_null() | std_c.is_null() | (std_c == 0)).then(1e-6).otherwise(std_c)
        expressions.append(
            ((pl.col(c).fill_null(mean_c) - mean_c) / std_c_safe_expr).fill_nan(0).fill_null(0).alias(f'{c}_ZSCORE')
        )
    df_copy = df_copy.with_columns(expressions)
    rank_cols = [f'{c}_RANK' for c in base_features]
    zscore_cols = [f'{c}_ZSCORE' for c in base_features]
    expressions = []

    # --- 4. Interactions (Original Set only) ---
    
    # Original Targeted Interactions (M4, M1, E1, V2)
    for c in ['M4','M1','E1','V2']:
        r = f'{c}_RANK'
        if f'{c}' in df_copy.columns and r in df_copy.columns:
             expressions.append((pl.col(c) * pl.col(r)).alias(f'{c}_x_{r}'))
             expressions.append((pl.col(c) / (pl.col(r) + 1e-6)).alias(f'{c}_div_{r}'))

    target_rank_cols = rank_cols[:12]
    # 1. Rank * Rank (Unique Pairs) - Adds 66 features
    for r_col1, r_col2 in product(target_rank_cols, target_rank_cols):
        c1 = r_col1.split('_')[0]
        c2 = r_col2.split('_')[0]
        if c1 < c2:
            expressions.append((pl.col(r_col1) * pl.col(r_col2)).fill_nan(0).fill_null(0).alias(f'{c1}R_x_{c2}R'))
    # 2. Rank * ZScore - Adds 9 features
    target_zscore_cols = zscore_cols[:9]
    for r_col, z_col in zip(rank_cols[:9], target_zscore_cols):
        expressions.append((pl.col(r_col) * pl.col(z_col)).fill_nan(0).fill_null(0).alias(f'{r_col}_x_{z_col}'))
        
    if expressions:
        df_copy = df_copy.with_columns(expressions)
    return df_copy

In [23]:

# ===========================================================================
# DATA LOADING AND SPLITTING
# ===========================================================================
logger.info("Loading data and splitting for validation...")
try:
    train_full_pl = pl.read_csv(DATA_PATH / "train.csv")
except FileNotFoundError:
    logger.error('Could not find \'train.csv\'. Please ensure the DATA_PATH is correct.')
    raise

train_full_pd = train_full_pl.to_pandas()

split_idx = int(len(train_full_pd) * 0.8)
train_pd = train_full_pd.head(split_idx).copy()
val_pd   = train_full_pd.tail(len(train_full_pd) - split_idx).copy()

# --- Revert to Classification Target (Required for Optimal Score) ---
train_pd['target_binary'] = (train_pd['market_forward_excess_returns'] > 0).astype(np.int8)
val_pd['target_binary'] = (val_pd['market_forward_excess_returns'] > 0).astype(np.int8)

train_pl = pl.from_pandas(train_pd)
val_pl   = pl.from_pandas(val_pd)

logger.info(f"✅ Oracle Dictionary ready. Binary Target created.")


2026-04-25 23:41:04,691 - Loading data and splitting for validation...
2026-04-25 23:41:04,922 - ✅ Oracle Dictionary ready. Binary Target created.


In [24]:
CBT_DEPTH = 11
CBT_LR = 0.03679836629440696
CBT_L2_REG = 4.601313084093791

# Fixed Estimators and Stopping
N_ESTIMATORS_MAX = 5000
EARLY_STOPPING_ROUNDS = 50

In [25]:
# ===========================================================================
# ENSEMBLE PIPELINE (MAIN PIPELINE)
# ===========================================================================
logger.info("Training ensemble (Expanded Feature Set)...")

# 1. Feature Engineering 
train_engineered_pl = create_features(train_pl)
val_engineered_pl = create_features(val_pl)

# 2. Feature Column Selection 
feature_cols = []
# NOTE: The list of suffixes reflects the restored original set (removed EWMA, Diff, Rank_Vol)
original_suffixes = ('_L1', '_L2', '_L5', '_L10', '_RMean5', '_RStd5', '_RMax5', '_RMean10', '_RStd10', '_RMax10', '_RANK', '_ZSCORE', '_x_RANK', '_div_RANK', '_x_R', '_x_ZSCORE')

for col in train_engineered_pl.columns:
    is_base_feature = col.startswith(('M', 'E', 'I', 'P', 'V', 'S'))
    is_engineered_feature = any(col.endswith(s) for s in original_suffixes)
    
    if (is_base_feature or is_engineered_feature) and (train_engineered_pl[col].null_count() / len(train_engineered_pl) < 0.95):
        feature_cols.append(col)

# Filter out targets and IDs
feature_cols = [c for c in feature_cols if c not in ['market_forward_excess_returns', 'target_binary', 'forward_returns', 'date_id']]
FINAL_FEATURE_COLS = feature_cols 

# NOTE: The feature count should return to approximately 1187 features
logger.info(f"Features: {len(FINAL_FEATURE_COLS)} (Restored Optimized Set)") 

# 3. Data Preparation for ML
X_train = train_engineered_pl.select(FINAL_FEATURE_COLS).fill_null(0).to_pandas()
y_train = train_pl['target_binary'].fill_null(0).to_numpy().astype(int)
X_val = val_engineered_pl.select(FINAL_FEATURE_COLS).fill_null(0).to_pandas()
y_val = val_pl['target_binary'].fill_null(0).to_numpy().astype(int)

# FIX: Enforce Feature Order for CatBoost
X_train = X_train.loc[:, FINAL_FEATURE_COLS]
X_val = X_val.loc[:, FINAL_FEATURE_COLS]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_lgb = pd.DataFrame(X_train_scaled, columns=FINAL_FEATURE_COLS)
X_val_lgb = pd.DataFrame(X_val_scaled, columns=FINAL_FEATURE_COLS)

# Create DMatrix objects for native XGBoost training
dtrain = xgb.DMatrix(X_train_scaled, label=y_train)
dval = xgb.DMatrix(X_val_scaled, label=y_val)


# 4. Model Training (3 Models + 1 inert model)
# --- WRAP MODEL TRAINING IN suppress_stderr() TO HIDE C-LEVEL WARNINGS ---
with suppress_stderr():

    # --- CatBoost ---
    cbt_model = CatBoostClassifier(
        iterations=N_ESTIMATORS_MAX, depth=int(CBT_DEPTH), learning_rate=CBT_LR, 
        l2_leaf_reg=CBT_L2_REG, random_seed=789, loss_function='Logloss', eval_metric='Logloss',
        bootstrap_type='Bayesian', task_type="GPU", logging_level='Silent', allow_writing_files=False, 
        early_stopping_rounds=EARLY_STOPPING_ROUNDS 
    ).fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
    )
    logger.info(f"CatBoostClassifier trained (Stopped at {cbt_model.get_best_iteration()} iterations).")

    # --- Logistic Regression (Trained, but weighted 0.0) ---
    logreg_model = LogisticRegression(
        penalty='l2', C=0.1, solver='liblinear', max_iter=1000, random_state=999
    ).fit(X_lgb, y_train)
    logger.info(f"LogisticRegression trained.")

logger.info("✅ Ensemble trained.")

2026-04-25 23:41:04,944 - Training ensemble (Expanded Feature Set)...
2026-04-25 23:41:11,947 - Features: 1187 (Restored Optimized Set)
2026-04-25 23:42:13,108 - CatBoostClassifier trained (Stopped at 37 iterations).
2026-04-25 23:42:16,979 - LogisticRegression trained.
2026-04-25 23:42:16,980 - ✅ Ensemble trained.


# Tabular transformer model

In [26]:
import tensorflow as tf
from tensorflow.keras import layers, models

def build_strong_tabular_model(num_features=len(FINAL_FEATURE_COLS)):
    inputs = layers.Input(shape=(num_features,))

    x = layers.LayerNormalization()(inputs)
    x = layers.Dense(512, activation="gelu")(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Dense(256, activation="gelu")(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Dense(128, activation="gelu")(x)
    x = layers.Dropout(0.2)(x)

    outputs = layers.Dense(1, activation="sigmoid")(x)

    model = models.Model(inputs, outputs)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
    )

    return model


In [27]:
model = build_strong_tabular_model(num_features=len(FINAL_FEATURE_COLS))

model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=64
)

Epoch 1/20
114/114 ━━━━━━━━━━━━━━━━━━━━ 7s 29ms/step - accuracy: 0.4927 - auc: 0.4991 - loss: 0.7592 - val_accuracy: 0.5149 - val_auc: 0.4993 - val_loss: 0.6973
Epoch 2/20
114/114 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5172 - auc: 0.5087 - loss: 0.7063 - val_accuracy: 0.5254 - val_auc: 0.5230 - val_loss: 0.6909
Epoch 3/20
114/114 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5265 - auc: 0.5328 - loss: 0.6957 - val_accuracy: 0.5271 - val_auc: 0.5002 - val_loss: 0.6921
Epoch 4/20
114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.5193 - auc: 0.5162 - loss: 0.6980 - val_accuracy: 0.4867 - val_auc: 0.5112 - val_loss: 0.6965
Epoch 5/20
114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.5083 - auc: 0.5089 - loss: 0.6958 - val_accuracy: 0.5276 - val_auc: 0.5096 - val_loss: 0.6963
Epoch 6/20
114/114 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5162 - auc: 0.5092 - loss: 0.6975 - val_accuracy: 0.5199 - val_auc: 0.5051 - val_loss: 0.6936
Epoch 7/20
114/114 ━━━━━━━━━━━━━━━━━━━━

In [28]:
# ===========================================================================
# VALIDATION – SCORE INTEGRATION (3-MODEL ENSEMBLE)
# ===========================================================================
logger.info('Preparing validation data for score logging (Based on 3-MODEL ENSEMBLE)...')
ML_CONF_FACTOR = 9.595373361542675 
prob_nn = model.predict(X_val).ravel()
prob_cbt = cbt_model.predict_proba(X_val)[:, 1]
# prob_logreg = logreg_model.predict_proba(X_val_lgb)[:, 1]
w_nn = 0.3620209418975796
w_cbt = 1- w_nn

avg_prob =  w_nn*prob_nn + w_cbt*prob_cbt

confidence = 2 * np.abs(avg_prob - 0.5)
positions_final = np.clip(confidence * ML_CONF_FACTOR, 0.0, 2.0)

submission_df_final = pd.DataFrame({'prediction': positions_final})

try:
    real_ps = score(val_pd, submission_df_final)
    logger.info(f'FINAL SUBMISSION PS SCORE ON VALIDATION (3-MODEL ENSEMBLE) = {real_ps:.6f}') 
except Exception as e:
    logger.error(f'Final Scoring error: {e}')
    real_ps = 0.0

2026-04-25 23:42:34,209 - Preparing validation data for score logging (Based on 3-MODEL ENSEMBLE)...


57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step


2026-04-25 23:42:34,871 - FINAL SUBMISSION PS SCORE ON VALIDATION (3-MODEL ENSEMBLE) = 0.550558


# Let's try to tune the hyperparameters

In [29]:
# prob_nn = model.predict(X_val, batch_size=2048).ravel()
# prob_cbt = cbt_model.predict_proba(X_val)[:, 1]

# assert prob_nn.shape == prob_cbt.shape

In [30]:
# import optuna
# import numpy as np

# def objective(trial):
#     # Hyperparameters to tune
#     ml_conf_factor = trial.suggest_float("ML_CONF_FACTOR", 0.5, 15.0, log=True)
#     w_nn = trial.suggest_float("w_nn", 0.0, 1.0)
#     w_cbt = 1.0 - w_nn

#     # Ensemble probabilities
#     avg_prob = w_nn * prob_nn + w_cbt * prob_cbt

#     # Confidence & positions
#     confidence = 2.0 * np.abs(avg_prob - 0.5)
#     positions_final = np.clip(confidence * ml_conf_factor, 0.0, 2.0)

#     submission_df = pd.DataFrame({"prediction": positions_final})

#     try:
#         ps = score(val_pd, submission_df)
#     except Exception:
#         ps = 0.0

#     # Optuna always MAXIMIZES here
#     return ps

In [31]:
# study = optuna.create_study(direction="maximize")

# study.optimize(
#     objective,
#     n_trials=100,
#     show_progress_bar=True
# )

In [32]:
# best_params = study.best_params
# best_score = study.best_value

# print("Best PS:", best_score)
# print("Best params:", best_params)

In [33]:
# ===========================================================================
# PREDICT FUNCTION (FINAL ROBUST VERSION - 3-MODEL ENSEMBLE)
# ===========================================================================
def predict(test: pl.DataFrame) -> float:
    
    ML_CONF_FACTOR = 9.595373361542675 
    
    # Check if all models are present (LogReg is needed for prediction, even if weighted 0)
    if logreg_model is None or scaler is None or not FINAL_FEATURE_COLS:
        return 0.0

    date_id = None
    try:
        date_id = int(test.select("date_id").to_series().item())
    except:
        pass
    
    try:
        test_engineered = create_features(test)
        X_test = test_engineered.select(FINAL_FEATURE_COLS).fill_null(0).to_pandas()
        
        X_clean = X_test.loc[:, FINAL_FEATURE_COLS]
        X_clean = X_clean.fillna(0).replace([np.inf, -np.inf], 0) 

        X_scaled = scaler.transform(X_clean.values)
        X_lgb = pd.DataFrame(X_scaled, columns=FINAL_FEATURE_COLS)
        dtest = xgb.DMatrix(X_scaled) 

       
        prob_nn = model.predict(X_clean).ravel()
        prob_cbt = cbt_model.predict_proba(X_clean)[:, 1][0]
        # prob_logreg = logreg_model.predict_proba(X_val_lgb)[:, 1]
        w_nn = 0.3620209418975796
        w_cbt = 1- w_nn
        
        avg_prob =  w_nn * prob_nn + w_cbt * prob_cbt

        confidence = 2 * abs(avg_prob - 0.5)
        position = np.clip(confidence * ML_CONF_FACTOR, 0, 2)

        return float(position)

    except Exception as e:
        logger.warning(f"ML Error (date_id: {date_id}): {e}")
        return 0.0

In [34]:
# ===========================================================================
# SERVER
# ===========================================================================
logger.info("Starting server")
import kaggle_evaluation.default_inference_server
inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway((str(DATA_PATH),))

logger.info("✅ Complete. ")

2026-04-25 23:42:34,946 - Starting server


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 239ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step


2026-04-25 23:42:37,672 - ✅ Complete. 


In [35]:
import pandas as pd
import numpy as np
import os

submission_path = '/kaggle/working/submission.parquet'

## 📊 SUBMISSION FILE VALIDATION

# Check if the file was successfully created
if not os.path.exists(submission_path):
    print(f"Validation Error: Submission file not found at {submission_path}")
    print("NOTE: This file is created during the 'Running local gateway for testing...' step.")
else:
    try:
        # Read the submission file
        df_sub = pd.read_parquet(submission_path)

        # --- Validation Checks ---

        # 1. Identify the prediction column (the single float column)
        float_cols = df_sub.select_dtypes(include=[np.float64]).columns

        if len(float_cols) == 1:
            prediction_col_name = float_cols[0]
            print(f"Column Check: Found single prediction column named '{prediction_col_name}'.")
        else:
            prediction_col_name = 'allocation' # Use standard name for range check fallback
            print(f"Column Check: Found {len(float_cols)} float columns. Using '{prediction_col_name}' for range check.")

        # 2. Check allocation range (0.0 to 2.0)
        if prediction_col_name in df_sub.columns:
            min_val = df_sub[prediction_col_name].min()
            max_val = df_sub[prediction_col_name].max()

            # Check if all values are between 0.0 and 2.0 (inclusive)
            if min_val >= 0.0 and max_val <= 2.0:
                range_check = "PASS"
            else:
                range_check = f"FAIL (Min: {min_val:.4f}, Max: {max_val:.4f})"

            print(f"Allocation Range Check (0.0 to 2.0): {range_check}")

        # 3. Display the file info
        print("\nFirst 10 Rows of Submission:")
        print(df_sub.head(10))

        print("\nSubmission Info:")
        df_sub.info()

    except Exception as e:
        print(f"Validation Error: Could not read or process the Parquet file. Error: {e}")

Column Check: Found single prediction column named 'prediction'.
Allocation Range Check (0.0 to 2.0): PASS

First 10 Rows of Submission:
   date_id  prediction
0     8980    0.060154
1     8981    0.008023
2     8982    0.552496
3     8983    0.016023
4     8984    0.005090
5     8985    0.388656
6     8986    0.173358
7     8987    0.103203
8     8988    0.476476
9     8989    0.154241

Submission Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date_id     10 non-null     int64  
 1   prediction  10 non-null     float64
dtypes: float64(1), int64(1)
memory usage: 292.0 bytes
